# <img src="../assets/Logo_03.png" class="course-logo" alt="Logo da disciplina"> EGC5310 — Estruturas de Dados<br><span class="week-label">SEMANA 05</span><br><span class="subtitle-custom">A chave mostra onde procurar · Hashing, dict, set e estruturas associativas</span>


## Notebook Estudante

As atividades desta semana exigem **previsão antes da execução**.

[**ABRIR O NOTEBOOK ESTUDANTE NO GOOGLE COLAB**](https://colab.research.google.com/github/professorviniciusramos/EGC5310-EstruturasDados/blob/main/Semana05/EGC5310-EstruturasDados-S05-04-Estudante.ipynb)

Arquivo: `EGC5310-EstruturasDados-S05-04-Estudante.ipynb`


## De onde estamos vindo?

- **S01:** procurar um a um → `O(n)`
- **S02–S03:** a representação e a operação determinam o custo
- **S04:** ordenar permite dividir o espaço → `O(log n)`

> Hoje: podemos usar a própria **chave** para indicar onde procurar?


## O preço da solução anterior

Busca binária é rápida, mas depende de uma lista ordenada pela mesma chave.

```python
matriculas_ordenadas.append(121)
matriculas_ordenadas.sort()
```

> Temos busca `O(log n)`. O problema está completamente resolvido?


In [ ]:
estudantes = [
    {"matricula": 20260123, "nome": "Ana", "curso": "CD"},
    {"matricula": 20260481, "nome": "Bruno", "curso": "CC"},
    {"matricula": 20260347, "nome": "Carla", "curso": "CD"},
]

for estudante in estudantes:
    print(estudante["matricula"], estudante["nome"])


## Problema orientador {.question-slide}

> **Como localizar uma matrícula rapidamente sem depender da posição do registro na coleção?**

Se a matrícula identifica exatamente um estudante, por que percorrer ou dividir uma coleção para encontrá-lo?


## Vamos inventar uma localização

Imagine dez posições: `0 1 2 3 4 5 6 7 8 9`.

```python
def hash_simples(matricula, tamanho=10):
    return matricula % tamanho
```

A matrícula é a **chave**. O resultado é uma **posição possível**.


## ATIVIDADE 1 — Prever posições

1. Abra o Notebook Estudante.
2. Calcule `matricula % 10` para `101`, `114`, `127` e `139`.
3. **Registre antes de executar.**

[**→ Abrir no Colab**](https://colab.research.google.com/github/professorviniciusramos/EGC5310-EstruturasDados/blob/main/Semana05/EGC5310-EstruturasDados-S05-04-Estudante.ipynb)


In [ ]:
def hash_simples(matricula, tamanho=10):
    return matricula % tamanho

for matricula in [101, 114, 127, 139]:
    print(matricula, "→ posição", hash_simples(matricula))


## Três ideias diferentes

| Ideia | Exemplo | Papel |
|---|---:|---|
| chave | `20260123` | identifica no domínio |
| função hash | `% 10` | transforma a chave |
| posição | `3` | indica onde procurar |

> A chave não é a posição. A posição é calculada a partir da chave.


## Primeira tabela hash {.smaller}

Uma primeira tentativa: uma posição guarda um item.

```python
tabela = [None] * 10

def inserir_sem_colisao(tabela, matricula, estudante):
    posicao = matricula % len(tabela)
    tabela[posicao] = (matricula, estudante)
```


In [ ]:
tabela_inicial = [None] * 10

def inserir_sem_colisao(tabela, matricula, estudante):
    posicao = matricula % len(tabela)
    tabela[posicao] = (matricula, estudante)

def buscar_sem_colisao(tabela, matricula):
    posicao = matricula % len(tabela)
    item = tabela[posicao]
    if item is not None and item[0] == matricula:
        return item[1]
    return None


## Inserir e recuperar {.smaller}

A função leva diretamente ao local em que o item **pode** estar.

```python
def buscar_sem_colisao(tabela, matricula):
    posicao = matricula % len(tabela)
    item = tabela[posicao]
    if item is not None and item[0] == matricula:
        return item[1]
    return None
```


In [ ]:
inserir_sem_colisao(tabela_inicial, 101, {"nome": "Ana"})
inserir_sem_colisao(tabela_inicial, 114, {"nome": "Bruno"})
print(tabela_inicial)
print(buscar_sem_colisao(tabela_inicial, 114))


## Vamos tentar quebrar a solução

```text
102 % 10 = 2
172 % 10 = 2
```

> O que acontecerá ao inserir os dois estudantes?


## ATIVIDADE 2 — Provocar uma colisão

1. Preveja o conteúdo da posição 2.
2. Execute as duas inserções.
3. Explique o que foi perdido — o programa não precisa lançar erro para estar incorreto.

[**→ Abrir no Colab**](https://colab.research.google.com/github/professorviniciusramos/EGC5310-EstruturasDados/blob/main/Semana05/EGC5310-EstruturasDados-S05-04-Estudante.ipynb)


In [ ]:
tabela_com_falha = [None] * 10
inserir_sem_colisao(tabela_com_falha, 102, {"nome": "Ana"})
inserir_sem_colisao(tabela_com_falha, 172, {"nome": "Bruno"})
print(tabela_com_falha[2])
print("buscar 102:", buscar_sem_colisao(tabela_com_falha, 102))


## Agora podemos nomear o problema {.concept-slide}

<span class="big-symbol">COLISÃO</span>

Duas chaves diferentes produzem a mesma posição.

> Colisão é esperada. O erro é não tratá-la.


## Uma coleção por posição

```python
tabela = [[] for _ in range(10)]
```

Cada posição contém um **bucket** independente. Chaves que colidem ficam encadeadas no mesmo bucket.

Evite `[[]] * 10`: essa expressão reutiliza a mesma lista interna.


## Inserção com encadeamento {.smaller}

```python
def inserir(tabela, matricula, estudante):
    posicao = matricula % len(tabela)
    bucket = tabela[posicao]
    for indice, (chave, _) in enumerate(bucket):
        if chave == matricula:
            bucket[indice] = (matricula, estudante)
            return
    bucket.append((matricula, estudante))
```


In [ ]:
def criar_tabela(tamanho=10):
    return [[] for _ in range(tamanho)]

def inserir(tabela, matricula, estudante):
    posicao = matricula % len(tabela)
    bucket = tabela[posicao]
    for indice, (chave_existente, _) in enumerate(bucket):
        if chave_existente == matricula:
            bucket[indice] = (matricula, estudante)
            return
    bucket.append((matricula, estudante))


## Busca com encadeamento {.smaller}

```python
def buscar(tabela, matricula):
    posicao = matricula % len(tabela)
    for chave, estudante in tabela[posicao]:
        if chave == matricula:
            return estudante
    return None
```


In [ ]:
def buscar(tabela, matricula):
    posicao = matricula % len(tabela)
    for chave_existente, estudante in tabela[posicao]:
        if chave_existente == matricula:
            return estudante
    return None

tabela = criar_tabela()
inserir(tabela, 102, {"nome": "Ana"})
inserir(tabela, 172, {"nome": "Bruno"})
print(tabela[2])
print(buscar(tabela, 102))


## ATIVIDADE 3 — Implementar inserção e busca

Complete as duas funções no Notebook Estudante e execute os testes.

Perguntas para conferir:

- Qual linha calcula o bucket?
- Por que ainda comparamos matrículas?
- O que deve ocorrer ao atualizar uma matrícula?

[**→ Abrir no Colab**](https://colab.research.google.com/github/professorviniciusramos/EGC5310-EstruturasDados/blob/main/Semana05/EGC5310-EstruturasDados-S05-04-Estudante.ipynb)


## O que o hashing mudou?

Antes: examinar a coleção ou dividir um intervalo ordenado.

Agora:

1. calcular uma posição a partir da chave;
2. examinar apenas o bucket correspondente;
3. confirmar a chave por causa das colisões.

> Muitas colisões tornam os buckets longos.


## Da tabela didática ao `dict`

No problema, queremos expressar diretamente:

```text
matrícula  →  registro do estudante
```

Python oferece uma estrutura associativa pronta: o `dict`.

```python
por_matricula = {}
for estudante in estudantes:
    por_matricula[estudante["matricula"]] = estudante

por_matricula.get(20260123)
```


In [ ]:
por_matricula = {}
for estudante in estudantes:
    por_matricula[estudante["matricula"]] = estudante

print(por_matricula[20260123])
print(por_matricula.get(99999999))


## ATIVIDADE 4 — Construir e consultar um `dict`

1. Construa `por_matricula` a partir da lista.
2. Consulte uma matrícula existente e uma ausente.
3. Preveja o efeito de inserir novamente uma chave já existente.

[**→ Abrir no Colab**](https://colab.research.google.com/github/professorviniciusramos/EGC5310-EstruturasDados/blob/main/Semana05/EGC5310-EstruturasDados-S05-04-Estudante.ipynb)


## Modelo didático × implementação real

Nossa tabela torna visíveis função, bucket e colisão. O `dict` real de Python é uma implementação otimizada e não é idêntico ao nosso encadeamento.

Podemos afirmar nesta semana:

- chaves são transformadas por hashing;
- colisões precisam ser tratadas;
- detalhes de capacidade e redimensionamento importam;
- consulta e inserção são tipicamente `O(1)` em média.


## Reforço visual opcional

Depois da implementação própria, observe o encadeamento em:

[**VisuAlgo — Hash Table**](https://visualgo.net/en/hashtable)

Sugestão: modo *Separate Chaining*; inserir `12`, `22`, `32` e buscar `22`.

> A animação reforça o mecanismo; não substitui a implementação.


## Novo requisito: apenas saber se já apareceu

Dados chegam de várias fontes. Precisamos detectar matrículas repetidas.

> Precisamos de `matrícula → estudante` ou apenas de pertencimento?

```python
processadas = set()
for matricula in matriculas_recebidas:
    if matricula in processadas:
        print("duplicada:", matricula)
    else:
        processadas.add(matricula)
```


In [ ]:
processadas = set()
for matricula in [102, 118, 102, 131, 118]:
    if matricula in processadas:
        print("duplicada:", matricula)
    else:
        processadas.add(matricula)


## ATIVIDADE 5 — Detectar duplicatas com `set`

Use `set` para produzir as matrículas duplicadas na ordem da segunda ocorrência.

Antes do código, escreva: por que `set` representa melhor este requisito que `dict`?

[**→ Abrir no Colab**](https://colab.research.google.com/github/professorviniciusramos/EGC5310-EstruturasDados/blob/main/Semana05/EGC5310-EstruturasDados-S05-04-Estudante.ipynb)


## Três estruturas, três intenções

| Estrutura | Expressa diretamente | Preserva repetições? | Acesso principal |
|---|---|:---:|---|
| `list` | sequência | sim | posição/percurso |
| `dict` | chave → valor | uma associação por chave | chave |
| `set` | pertencimento | não | elemento |

> A melhor escolha depende da operação predominante.


## ATIVIDADE 6 — Escolher e justificar

Escolha `list`, `dict` ou `set` para cada caso:

1. guardar estudantes na ordem de chegada;
2. localizar estudante pela matrícula;
3. impedir matrícula duplicada;
4. associar código de disciplina a informações.

A justificativa deve citar a **operação predominante**.

[**→ Abrir no Colab**](https://colab.research.google.com/github/professorviniciusramos/EGC5310-EstruturasDados/blob/main/Semana05/EGC5310-EstruturasDados-S05-04-Estudante.ipynb)


## Três mecanismos de recuperação

| Estratégia | Mecanismo | Busca típica |
|---|---|---:|
| sequencial | examinar sucessivamente | `O(n)` |
| binária | descartar metade ordenada | `O(log n)` |
| hashing | calcular posição + bucket curto | `O(1)` médio/esperado |


## `O(1)` não quer dizer...

- uma única instrução;
- tempo zero;
- ausência de colisões;
- garantia de mesmo tempo em qualquer entrada.

> Sob condições adequadas, o custo médio dominante não cresce proporcionalmente a `n`.


## E se todas as chaves colidirem?

```text
bucket 2 → 102 → 172 → 242 → 312 → ...
```

A busca volta a percorrer uma coleção que pode crescer com `n`.

Distribuição das chaves, capacidade, colisões e redimensionamento importam.


## ATIVIDADE 7 — Interpretar complexidade

Corrija em duas frases:

> “`O(1)` significa que o `dict` sempre encontra o valor com uma única instrução e nunca tem colisões.”

Use obrigatoriamente as expressões **médio/esperado** e **crescimento**.

[**→ Abrir no Colab**](https://colab.research.google.com/github/professorviniciusramos/EGC5310-EstruturasDados/blob/main/Semana05/EGC5310-EstruturasDados-S05-04-Estudante.ipynb)


## Síntese da Semana 05

1. Uma chave pode orientar diretamente a localização.
2. Colisões são inevitáveis e precisam de tratamento.
3. `dict` expressa chave → valor.
4. `set` expressa pertencimento e unicidade.
5. Hashing oferece `O(1)` médio/esperado sob condições adequadas.
6. Estruturas são escolhidas pelas operações e restrições do problema.


## Ponte para a Semana 06 {.question-slide}

> **Qual estratégia é mais rápida na prática?**

```python
buscar_lista(...)
buscar_binaria(...)
por_matricula.get(...)
```

Precisaremos controlar tamanho, buscas existentes/ausentes, preparação, repetições e métrica.
